# 03 - Explainable AI and Retention Recommendations

**Owner:** Gurnoor

**Scope:** This notebook does not retrain, re-tune, or re-select the churn model. It loads the model artifact produced in `02_modeling.ipynb` (final tuned XGBoost, decision threshold = 0.30) and builds the explainability and retention-recommendation layer on top of it:

Customer Data -> Preprocessing -> Churn Prediction -> Churn Probability -> SHAP Explanation -> Human-Readable Churn Reasons -> Retention Recommendation -> (Dashboard, owned by Ujjwal)

**Status:** Scaffold only. Sections are filled in incrementally, one at a time, and each stage is tested before moving to the next.

## 1. Load Model and Configuration

**Finding (before writing any code below):** `models/` originally contained only `model_config.json` (`{"threshold": 0.3}`). There was no `churn_model.pkl` / `xgboost_model.pkl` committed to the repo, because `models/*.pkl` is gitignored by design, and `02_modeling.ipynb` builds `final_pipeline = Pipeline([("preprocessor", preprocessor), ("model", xgb_model)])` and fits it (cell 25) but never calls `joblib.dump` on it — only the raw `xgb_model` (fit on already-preprocessed arrays) gets saved, to `../models/xgboost_model.pkl`, which is also gitignored and therefore not present locally either.

**Resolution used here:** rather than inventing a model or skipping ahead, the exact steps from `02_modeling.ipynb` were reproduced line-for-line (same 20 features, same `Total Charges` numeric coercion, same `train_test_split(test_size=0.20, random_state=42, stratify=y)`, same `ColumnTransformer` (`StandardScaler` on 4 numeric features, `OneHotEncoder(handle_unknown="ignore")` on 16 categorical features), same `XGBClassifier` hyperparameters) and the resulting `final_pipeline` was persisted to `models/churn_model.pkl`. This is **not** a new/retrained model — it is Gracy's exact pinned configuration, just actually saved to disk (which her notebook currently omits).

**Verification against Gracy's reported numbers:**
- Train/test shapes match exactly: 5634 / 1409 rows, 20 raw features -> 47 processed features.
- At threshold 0.30 on the test set: Accuracy ≈ 0.762, Precision(churn) ≈ 0.54, Recall(churn) ≈ 0.78, F1(churn) ≈ 0.63, churners predicted = 541/1409 — matches `results/threshold_analysis.csv` (0.5399 / 0.7781 / 0.6375 at threshold 0.30) and the reported ~538/1409 within rounding/library-version noise.
- At the default 0.50 threshold: Accuracy ≈ 0.807, ROC-AUC ≈ 0.857, vs. Gracy's reported 0.8034 / 0.8555 in `results/model_comparison.csv` — a small (~0.3-1pt) discrepancy, most likely from `xgboost`/`scikit-learn` minor version differences (this environment: xgboost 3.2.0, scikit-learn 1.9.0) rather than a different model. **This notebook does not overwrite `results/model_comparison.csv` or any other existing result file** — those numbers remain Gracy's official reported metrics. Anywhere this notebook restates them, it cites the file, not a re-measurement.

**Artifact structure:** `models/churn_model.pkl` is a full `sklearn.pipeline.Pipeline` with two named steps — `"preprocessor"` (the `ColumnTransformer`) and `"model"` (the fitted `XGBClassifier`) — so it accepts **raw, unprocessed customer records** (the 20 original feature columns) directly; preprocessing does not need to be reapplied by hand before calling `.predict()` / `.predict_proba()`. This matters for the SHAP setup in Section 4: SHAP's `TreeExplainer` needs the trees' native input space, so it will be pointed at `pipeline.named_steps["model"]` and fed data already run through `pipeline.named_steps["preprocessor"].transform(...)`, not the raw pipeline itself.

### 1a. One-time artifact regeneration (reproduces Gracy's pipeline; run only if `models/churn_model.pkl` is missing)

This cell is **not** the XAI workflow itself — it is a stopgap that reproduces the exact preprocessing + XGBoost configuration from `02_modeling.ipynb` and persists it, because that notebook currently fits `final_pipeline` but never saves it. Ideally this save step gets added directly to `02_modeling.ipynb` by Gracy so this cell becomes unnecessary; until then it is guarded to skip if the artifact already exists, so it never silently retrains over a real saved model.

In [1]:
import os
import joblib
import pandas as pd

MODEL_PATH = "../models/churn_model.pkl"
DATA_PATH = "../data/Telco_customer_churn.xlsx"

if not os.path.exists(MODEL_PATH):
    from sklearn.model_selection import train_test_split
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.pipeline import Pipeline
    from xgboost import XGBClassifier

    df = pd.read_excel(DATA_PATH)

    features = [
        "Gender", "Senior Citizen", "Partner", "Dependents", "Tenure Months",
        "Phone Service", "Multiple Lines", "Internet Service", "Online Security",
        "Online Backup", "Device Protection", "Tech Support", "Streaming TV",
        "Streaming Movies", "Contract", "Paperless Billing", "Payment Method",
        "Monthly Charges", "Total Charges", "CLTV",
    ]
    numeric_features = ["Tenure Months", "Monthly Charges", "Total Charges", "CLTV"]
    categorical_features = [f for f in features if f not in numeric_features]

    df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")
    X = df[features]
    y = df["Churn Value"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    preprocessor = ColumnTransformer(transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop=None), categorical_features),
    ])

    xgb_model = XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8,
        colsample_bytree=1.0, min_child_weight=1, eval_metric="logloss",
        random_state=42, n_jobs=-1,
    )

    final_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", xgb_model),
    ])
    final_pipeline.fit(X_train, y_train)

    joblib.dump(final_pipeline, MODEL_PATH)
    print(f"Regenerated and saved pipeline to {MODEL_PATH}")
else:
    print(f"{MODEL_PATH} already exists — skipping regeneration.")

../models/churn_model.pkl already exists — skipping regeneration.


### 1b. Load the model artifact and threshold config (actual Section 1 workflow)

In [2]:
import json

CONFIG_PATH = "../models/model_config.json"

churn_pipeline = joblib.load(MODEL_PATH)

with open(CONFIG_PATH) as f:
    model_config = json.load(f)

CHURN_THRESHOLD = model_config["threshold"]

print("Loaded object type:", type(churn_pipeline))
print("Pipeline steps:", list(churn_pipeline.named_steps.keys()))
print("Preprocessor:", churn_pipeline.named_steps["preprocessor"])
print("Model:", type(churn_pipeline.named_steps["model"]))
print("Decision threshold (from model_config.json):", CHURN_THRESHOLD)

Loaded object type: <class 'sklearn.pipeline.Pipeline'>
Pipeline steps: ['preprocessor', 'model']
Preprocessor: ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['Tenure Months', 'Monthly Charges',
                                  'Total Charges', 'CLTV']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['Gender', 'Senior Citizen', 'Partner',
                                  'Dependents', 'Phone Service',
                                  'Multiple Lines', 'Internet Service',
                                  'Online Security', 'Online Backup',
                                  'Device Protection', 'Tech Support',
                                  'Streaming TV', 'Streaming Movies',
                                  'Contract', 'Paperless Billing',
                                  'Payment Method'])])
Model: <class 'xgboost.sklearn.XGBClassifier'>
Decision threshold (from mode

## 2. Load / Prepare Customer Data

This section prepares the customer records the rest of the notebook will explain. It reuses the **same train/test split** as `02_modeling.ipynb` (`test_size=0.20, random_state=42, stratify=y`) so that the customers used for XAI examples are genuinely held-out test customers, not rows the model was trained on. `CustomerID` is kept alongside the 20 modeling features purely for display/lookup — it is never passed into the pipeline as a feature.

In [3]:
from sklearn.model_selection import train_test_split

FEATURES = [
    "Gender", "Senior Citizen", "Partner", "Dependents", "Tenure Months",
    "Phone Service", "Multiple Lines", "Internet Service", "Online Security",
    "Online Backup", "Device Protection", "Tech Support", "Streaming TV",
    "Streaming Movies", "Contract", "Paperless Billing", "Payment Method",
    "Monthly Charges", "Total Charges", "CLTV",
]

df = pd.read_excel(DATA_PATH)
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")

X = df[FEATURES]
y = df["Churn Value"]
customer_ids = df["CustomerID"]

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, customer_ids, test_size=0.20, random_state=42, stratify=y
)

test_customers = X_test.copy()
test_customers.insert(0, "CustomerID", id_test)
test_customers["Churn Value"] = y_test

print("Held-out test customers:", test_customers.shape[0])
test_customers.head()

Held-out test customers: 1409


,CustomerID,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,...,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,CLTV,Churn Value
2196,4376-KFVRS,Male,No,Yes,No,72,Yes,Yes,Fiber optic,Yes,...,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),114.05,8468.20,4842,0
3549,2754-SDJRD,Female,Yes,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,Yes,Yes,Month-to-month,Yes,Credit card (automatic),100.15,908.55,5157,0
3515,9917-KWRBE,Female,No,Yes,No,41,Yes,Yes,DSL,Yes,...,No,Yes,No,One year,Yes,Credit card (automatic),78.35,3211.20,2894,0
5162,0365-GXEZS,Male,No,Yes,No,18,Yes,No,Fiber optic,No,...,Yes,No,No,Month-to-month,No,Electronic check,78.20,1468.75,2831,0
4642,9385-NXKDA,Female,No,Yes,No,72,Yes,Yes,DSL,Yes,...,No,Yes,Yes,Two year,Yes,Credit card (automatic),82.65,5919.35,4324,0


## 3. Generate Churn Predictions

`predict_customer()` is the core prediction function the rest of the notebook (and eventually Ujjwal's `explain_customer()` API output) builds on. It takes one raw customer record (the 20 modeling features, no `CustomerID`), runs it through the loaded pipeline, and applies the project's fixed 0.30 decision threshold — it does not use scikit-learn's default 0.50 cutoff.

In [4]:
def predict_customer(customer_features: pd.DataFrame) -> dict:
    """Predict churn probability/decision for one raw customer record.

    customer_features: single-row DataFrame with the 20 modeling columns
    (same schema as FEATURES), no CustomerID or target column.
    """
    probability = churn_pipeline.predict_proba(customer_features)[:, 1][0]
    is_churn_risk = int(probability >= CHURN_THRESHOLD)
    return {
        "churn_probability": float(probability),
        "churn_prediction": is_churn_risk,
    }


# Sanity check: pick one held-out test customer and predict.
sample_customer_id = test_customers.iloc[0]["CustomerID"]
sample_row = X_test.loc[[test_customers.index[0]]]

result = predict_customer(sample_row)

print("CustomerID:", sample_customer_id)
print("Churn probability: {:.2%}".format(result["churn_probability"]))
print("Predicted class (threshold {:.2f}):".format(CHURN_THRESHOLD), result["churn_prediction"])
print("Actual churn label:", int(test_customers.iloc[0]["Churn Value"]))

CustomerID: 4376-KFVRS
Churn probability: 3.30%
Predicted class (threshold 0.30): 0
Actual churn label: 0


## 4. SHAP Setup

## 5. Global Explainability

## 6. Local Explainability

## 7. Human-Readable Explanations

## 8. Churn Driver Classification

## 9. Retention Recommendation Logic

## 10. Example Customer Explanations

## 11. XAI Validation / Sanity Checks

## 12. Conclusion